# 2.0 - Data Qualiy with OpenAI ChatGPT 3.5-Turbo

In [1]:
# import
import os
import json
import pandas as pd
from openai import OpenAI
MODEL = os.getenv('MODEL')
API_KEY = os.getenv('CHATGPT_TOKEN')

In [2]:
# setup
client = OpenAI(api_key=API_KEY) #initialize the OpenAI client

##### OpenAI documentation: https://platform.openai.com/docs/guides/batch?lang=python

In [ ]:
#define batch input file
batch_input_file = client.files.create(
    file = open("../data/processed/requests.jsonl", "rb"),
    purpose = "batch"
)

In [ ]:
batch_input_file_id = batch_input_file.id
# create the batch process
batch_object = client.batches.create(
    input_file_id = batch_input_file_id,
    endpoint = "/v1/chat/completions",
    completion_window = "24h",
    metadata = {
        "description": "nightly eval job"
    }
)
print(batch_object.id) # save the batch id for reference on the .env file as BATCH_PROCESS_ID

---

In [3]:
# read the batch id from the .env file
batch_object_id = os.getenv('BATCH_PROCESS_ID')

In [ ]:
# retrieve data from batch process
batch = client.batches.retrieve(batch_object_id)
print(f'Batch Object id: {batch.id}')
print(f'Status: {batch.status} - {batch.request_counts}')
print(f'Batch file: {batch.output_file_id}')

In [68]:
# get the output file
file_response = client.files.content(batch.output_file_id)

In [69]:
# save results to a jsonl file
json_list = list()
with open('../data/processed/responses.jsonl', 'w') as f:
    for response in [x for x in file_response.text.split('\n') if x]:
        json.dump(json.loads(response), f)
        json_file = json.loads(response)
        current_json = {
            'text' : json_file['response']['body']['choices'][0]['message']['content'],
            'doc_id' : json_file['custom_id']
        }
        json_list.append(current_json.copy())
        f.write('\n')

---
# Retrieve data and add cleaned article texts

In [104]:
df_raw = pd.read_parquet('../data/processed/full_articles_data.parquet', engine = 'pyarrow')
df_cleaned = pd.DataFrame(json_list)

In [ ]:
# define columns and renaming
df_raw['doc_id'] = "doc_" + df_raw['index'].astype(str)
df_raw['title'] = df_raw['object_title']
df_raw['category'] = df_raw['domain_category']
df_raw['post type'] = df_raw['object_post_type']
df_raw['url'] = df_raw['object_url']
df_raw['links words rate'] = df_raw['links_words_rate']
df_raw['publication date'] = pd.to_datetime(df_raw['object_pub_date'], format = '%Y-%m-%d %H:%M:%S').dt.date

In [ ]:
# merge the dataframes with the text cleaned by ChatGPT
out_df = df_raw.merge(df_cleaned, on = 'doc_id', how = 'left')

In [ ]:
# define output dataframe structure
out_df = out_df[['doc_id', 'title', 'publication date', 'category', 'post type', 'domain', 'url', 'text', 'links words rate' ]]

In [117]:
# add new columns with the text chunked by sentence
out_df['text chunked'] = out_df['text'].apply(lambda x: [content.strip() for content in x.replace('\n', '').split('|') if content])

In [ ]:
# save the output dataframe
out_df.to_parquet('../data/processed/article_texts_cleaned.parquet', index=False, engine = 'pyarrow')

In [ ]:
#client.batches.cancel(batch_object_id) # cancel the batch process (raise an error if the batch is already completed)